# Практикум 10. Синтаксический парсинг: деревья зависимостей и деревья составляющих

В лаборатории 10 вы рассматривали дуги зависимостей, пошагово запускали парсер arc-standard и заполняли таблицу CYK. Здесь то же самое в коде:

1. запишем деревья зависимостей в формате CoNLL-U, нарисуем дуги и сравним порядок слов в русском (SVO) и узбекском (SOV);
2. реализуем систему переходов arc-standard: оракул, трассу стека и буфера, затем обучим классификатор действий и получим настоящий (пусть маленький) парсер;
3. напишем алгоритм CYK для грамматики в нормальной форме Хомского и восстановим все деревья неоднозначного предложения «Я видел девушку с биноклем»;
4. сравним результат с `nltk.ChartParser`, выведем деревья через `nltk.Tree` и выберем лучшее дерево вероятностной грамматикой (PCFG).

*Практикум к лаборатории 10 курса «Алгоритмы NLP». Запускайте ячейки сверху вниз: Shift+Enter или «Среда выполнения → Выполнить всё». Чтобы сохранить свои ответы, сделайте «Файл → Сохранить копию на Диске».*

## 1. Деревья зависимостей и формат CoNLL-U

В дереве зависимостей каждое слово, кроме корня, зависит ровно от одного слова — своей вершины (head). На дуге написан тип связи (Universal Dependencies): `nsubj` — подлежащее, `obj` — прямое дополнение, `iobj` — косвенное дополнение, `obl` — обстоятельство с падежом или предлогом, `amod` — определение-прилагательное, `det` — местоименное определение, `nmod` — именное определение, `case` — предлог при существительном, `advmod` — наречие.

Формат CoNLL-U: одно слово на строку, 10 столбцов: ID, FORM (словоформа), LEMMA, UPOS, XPOS, FEATS, HEAD (номер вершины, 0 — корень), DEPREL, DEPS, MISC. Ниже маленький трибанк, размеченный для этого практикума: 19 обучающих и 6 тестовых предложений. Неиспользуемые столбцы заполнены `_`. Для сравнения порядка слов добавлены те же предложения по-узбекски.

Ниже трибанк читается, и первое предложение (пример из лаборатории) рисуется вместе с узбекским переводом «Talaba kutubxonada qiziqarli kitob oʻqiyapti».

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter, defaultdict

TRAIN_CONLLU = """
# sent_id = 1
# text = Студент читает интересную книгу в библиотеке
1  Студент     студент     NOUN  _ _ 2  nsubj     _ _
2  читает      читать      VERB  _ _ 0  root      _ _
3  интересную  интересный  ADJ   _ _ 4  amod      _ _
4  книгу       книга       NOUN  _ _ 2  obj       _ _
5  в           в           ADP   _ _ 6  case      _ _
6  библиотеке  библиотека  NOUN  _ _ 2  obl       _ _

# sent_id = 2
# text = Мама готовит вкусный плов
1  Мама     мама      NOUN  _ _ 2  nsubj     _ _
2  готовит  готовить  VERB  _ _ 0  root      _ _
3  вкусный  вкусный   ADJ   _ _ 4  amod      _ _
4  плов     плов      NOUN  _ _ 2  obj       _ _

# sent_id = 3
# text = Мой брат живёт в Ташкенте
1  Мой       мой      DET   _ _ 2  det       _ _
2  брат      брат     NOUN  _ _ 3  nsubj     _ _
3  живёт     жить     VERB  _ _ 0  root      _ _
4  в         в        ADP   _ _ 5  case      _ _
5  Ташкенте  Ташкент  PROPN _ _ 3  obl       _ _

# sent_id = 4
# text = Дети играют во дворе
1  Дети    ребёнок  NOUN  _ _ 2  nsubj     _ _
2  играют  играть   VERB  _ _ 0  root      _ _
3  во      в        ADP   _ _ 4  case      _ _
4  дворе   двор     NOUN  _ _ 2  obl       _ _

# sent_id = 5
# text = Учитель дал ученикам новые книги
1  Учитель   учитель  NOUN  _ _ 2  nsubj     _ _
2  дал       дать     VERB  _ _ 0  root      _ _
3  ученикам  ученик   NOUN  _ _ 2  iobj      _ _
4  новые     новый    ADJ   _ _ 5  amod      _ _
5  книги     книга    NOUN  _ _ 2  obj       _ _

# sent_id = 6
# text = Я вчера купил новую машину
1  Я       я       PRON  _ _ 3  nsubj     _ _
2  вчера   вчера   ADV   _ _ 3  advmod    _ _
3  купил   купить  VERB  _ _ 0  root      _ _
4  новую   новый   ADJ   _ _ 5  amod      _ _
5  машину  машина  NOUN  _ _ 3  obj       _ _

# sent_id = 7
# text = Сестра пишет письмо подруге
1  Сестра   сестра   NOUN  _ _ 2  nsubj     _ _
2  пишет    писать   VERB  _ _ 0  root      _ _
3  письмо   письмо   NOUN  _ _ 2  obj       _ _
4  подруге  подруга  NOUN  _ _ 2  iobj      _ _

# sent_id = 8
# text = Отец работает на большом заводе
1  Отец      отец      NOUN  _ _ 2  nsubj     _ _
2  работает  работать  VERB  _ _ 0  root      _ _
3  на        на        ADP   _ _ 5  case      _ _
4  большом   большой   ADJ   _ _ 5  amod      _ _
5  заводе    завод     NOUN  _ _ 2  obl       _ _

# sent_id = 9
# text = Птицы летят на юг
1  Птицы  птица   NOUN  _ _ 2  nsubj     _ _
2  летят  лететь  VERB  _ _ 0  root      _ _
3  на     на      ADP   _ _ 4  case      _ _
4  юг     юг      NOUN  _ _ 2  obl       _ _

# sent_id = 10
# text = Я пошёл в парк с другом
1  Я       я      PRON  _ _ 2  nsubj     _ _
2  пошёл   пойти  VERB  _ _ 0  root      _ _
3  в       в      ADP   _ _ 4  case      _ _
4  парк    парк   NOUN  _ _ 2  obl       _ _
5  с       с      ADP   _ _ 6  case      _ _
6  другом  друг   NOUN  _ _ 2  obl       _ _

# sent_id = 11
# text = Бабушка рассказала детям интересную сказку
1  Бабушка     бабушка     NOUN  _ _ 2  nsubj     _ _
2  рассказала  рассказать  VERB  _ _ 0  root      _ _
3  детям       ребёнок     NOUN  _ _ 2  iobj      _ _
4  интересную  интересный  ADJ   _ _ 5  amod      _ _
5  сказку      сказка      NOUN  _ _ 2  obj       _ _

# sent_id = 12
# text = Студенты сдали трудный экзамен
1  Студенты  студент  NOUN  _ _ 2  nsubj     _ _
2  сдали     сдать    VERB  _ _ 0  root      _ _
3  трудный   трудный  ADJ   _ _ 4  amod      _ _
4  экзамен   экзамен  NOUN  _ _ 2  obj       _ _

# sent_id = 13
# text = Кошка спит на диване
1  Кошка   кошка  NOUN  _ _ 2  nsubj     _ _
2  спит    спать  VERB  _ _ 0  root      _ _
3  на      на     ADP   _ _ 4  case      _ _
4  диване  диван  NOUN  _ _ 2  obl       _ _

# sent_id = 14
# text = Мальчик быстро решил задачу
1  Мальчик  мальчик  NOUN  _ _ 3  nsubj     _ _
2  быстро   быстро   ADV   _ _ 3  advmod    _ _
3  решил    решить   VERB  _ _ 0  root      _ _
4  задачу   задача   NOUN  _ _ 3  obj       _ _

# sent_id = 15
# text = Мы читаем книгу
1  Мы      мы      PRON  _ _ 2  nsubj     _ _
2  читаем  читать  VERB  _ _ 0  root      _ _
3  книгу   книга   NOUN  _ _ 2  obj       _ _

# sent_id = 16
# text = Врач осмотрел больного ребёнка
1  Врач      врач       NOUN  _ _ 2  nsubj     _ _
2  осмотрел  осмотреть  VERB  _ _ 0  root      _ _
3  больного  больной    ADJ   _ _ 4  amod      _ _
4  ребёнка   ребёнок    NOUN  _ _ 2  obj       _ _

# sent_id = 17
# text = Ахмад купил хлеб в магазине
1  Ахмад     Ахмад    PROPN _ _ 2  nsubj     _ _
2  купил     купить   VERB  _ _ 0  root      _ _
3  хлеб      хлеб     NOUN  _ _ 2  obj       _ _
4  в         в        ADP   _ _ 5  case      _ _
5  магазине  магазин  NOUN  _ _ 2  obl       _ _

# sent_id = 18
# text = Девочка подарила маме красивые цветы
1  Девочка   девочка   NOUN  _ _ 2  nsubj     _ _
2  подарила  подарить  VERB  _ _ 0  root      _ _
3  маме      мама      NOUN  _ _ 2  iobj      _ _
4  красивые  красивый  ADJ   _ _ 5  amod      _ _
5  цветы     цветок    NOUN  _ _ 2  obj       _ _

# sent_id = 19
# text = Книга моего брата лежит на столе
1  Книга  книга   NOUN  _ _ 4  nsubj     _ _
2  моего  мой     DET   _ _ 3  det       _ _
3  брата  брат    NOUN  _ _ 1  nmod      _ _
4  лежит  лежать  VERB  _ _ 0  root      _ _
5  на     на      ADP   _ _ 6  case      _ _
6  столе  стол    NOUN  _ _ 4  obl       _ _
"""

TEST_CONLLU = """
# sent_id = 1
# text = Брат купил интересную книгу
1  Брат        брат        NOUN  _ _ 2  nsubj     _ _
2  купил       купить      VERB  _ _ 0  root      _ _
3  интересную  интересный  ADJ   _ _ 4  amod      _ _
4  книгу       книга       NOUN  _ _ 2  obj       _ _

# sent_id = 2
# text = Студенты читают в библиотеке
1  Студенты    студент     NOUN  _ _ 2  nsubj     _ _
2  читают      читать      VERB  _ _ 0  root      _ _
3  в           в           ADP   _ _ 4  case      _ _
4  библиотеке  библиотека  NOUN  _ _ 2  obl       _ _

# sent_id = 3
# text = Учитель написал письмо директору
1  Учитель    учитель   NOUN  _ _ 2  nsubj     _ _
2  написал    написать  VERB  _ _ 0  root      _ _
3  письмо     письмо    NOUN  _ _ 2  obj       _ _
4  директору  директор  NOUN  _ _ 2  iobj      _ _

# sent_id = 4
# text = Дети пьют холодную воду
1  Дети      ребёнок   NOUN  _ _ 2  nsubj     _ _
2  пьют      пить      VERB  _ _ 0  root      _ _
3  холодную  холодный  ADJ   _ _ 4  amod      _ _
4  воду      вода      NOUN  _ _ 2  obj       _ _

# sent_id = 5
# text = Отец быстро прочитал газету
1  Отец      отец       NOUN  _ _ 3  nsubj     _ _
2  быстро    быстро     ADV   _ _ 3  advmod    _ _
3  прочитал  прочитать  VERB  _ _ 0  root      _ _
4  газету    газета     NOUN  _ _ 3  obj       _ _

# sent_id = 6
# text = Сестра моего друга работает в больнице
1  Сестра    сестра    NOUN  _ _ 4  nsubj     _ _
2  моего     мой       DET   _ _ 3  det       _ _
3  друга     друг      NOUN  _ _ 1  nmod      _ _
4  работает  работать  VERB  _ _ 0  root      _ _
5  в         в         ADP   _ _ 6  case      _ _
6  больнице  больница  NOUN  _ _ 4  obl       _ _
"""

# те же предложения по-узбекски — для сравнения порядка слов (раздел 1)
OTHER_CONLLU = """
# sent_id = 1
# text = Talaba kutubxonada qiziqarli kitob oʻqiyapti
1  Talaba       talaba     NOUN  _ _ 5  nsubj     _ _
2  kutubxonada  kutubxona  NOUN  _ _ 5  obl       _ _
3  qiziqarli    qiziqarli  ADJ   _ _ 4  amod      _ _
4  kitob        kitob      NOUN  _ _ 5  obj       _ _
5  oʻqiyapti    oʻqi       VERB  _ _ 0  root      _ _

# sent_id = 2
# text = Onam mazali palov pishiryapti
1  Onam         ona     NOUN  _ _ 4  nsubj     _ _
2  mazali       mazali  ADJ   _ _ 3  amod      _ _
3  palov        palov   NOUN  _ _ 4  obj       _ _
4  pishiryapti  pishir  VERB  _ _ 0  root      _ _

# sent_id = 3
# text = Akam Toshkentda yashaydi
1  Akam        aka       NOUN  _ _ 3  nsubj     _ _
2  Toshkentda  Toshkent  PROPN _ _ 3  obl       _ _
3  yashaydi    yasha     VERB  _ _ 0  root      _ _

# sent_id = 4
# text = Bolalar hovlida oʻynayapti
1  Bolalar     bola   NOUN  _ _ 3  nsubj     _ _
2  hovlida     hovli  NOUN  _ _ 3  obl       _ _
3  oʻynayapti  oʻyna  VERB  _ _ 0  root      _ _

# sent_id = 5
# text = Oʻqituvchi oʻquvchilarga yangi kitoblar berdi
1  Oʻqituvchi     oʻqituvchi  NOUN  _ _ 5  nsubj     _ _
2  oʻquvchilarga  oʻquvchi    NOUN  _ _ 5  obl       _ _
3  yangi          yangi       ADJ   _ _ 4  amod      _ _
4  kitoblar       kitob       NOUN  _ _ 5  obj       _ _
5  berdi          ber         VERB  _ _ 0  root      _ _

# sent_id = 6
# text = Men kecha yangi mashina oldim
1  Men      men      PRON  _ _ 5  nsubj     _ _
2  kecha    kecha    ADV   _ _ 5  advmod    _ _
3  yangi    yangi    ADJ   _ _ 4  amod      _ _
4  mashina  mashina  NOUN  _ _ 5  obj       _ _
5  oldim    ol       VERB  _ _ 0  root      _ _

# sent_id = 7
# text = Singlim dugonasiga xat yozyapti
1  Singlim     singil  NOUN  _ _ 4  nsubj     _ _
2  dugonasiga  dugona  NOUN  _ _ 4  obl       _ _
3  xat         xat     NOUN  _ _ 4  obj       _ _
4  yozyapti    yoz     VERB  _ _ 0  root      _ _

# sent_id = 8
# text = Otam katta zavodda ishlaydi
1  Otam      ota    NOUN  _ _ 4  nsubj     _ _
2  katta     katta  ADJ   _ _ 3  amod      _ _
3  zavodda   zavod  NOUN  _ _ 4  obl       _ _
4  ishlaydi  ishla  VERB  _ _ 0  root      _ _

# sent_id = 9
# text = Qushlar janubga uchyapti
1  Qushlar   qush   NOUN  _ _ 3  nsubj     _ _
2  janubga   janub  NOUN  _ _ 3  obl       _ _
3  uchyapti  uch    VERB  _ _ 0  root      _ _

# sent_id = 10
# text = Men doʻstim bilan parkka bordim
1  Men      men    PRON  _ _ 5  nsubj     _ _
2  doʻstim  doʻst  NOUN  _ _ 5  obl       _ _
3  bilan    bilan  ADP   _ _ 2  case      _ _
4  parkka   park   NOUN  _ _ 5  obl       _ _
5  bordim   bor    VERB  _ _ 0  root      _ _

# sent_id = 11
# text = Buvim bolalarga qiziqarli ertak aytdi
1  Buvim      buvi       NOUN  _ _ 5  nsubj     _ _
2  bolalarga  bola       NOUN  _ _ 5  obl       _ _
3  qiziqarli  qiziqarli  ADJ   _ _ 4  amod      _ _
4  ertak      ertak      NOUN  _ _ 5  obj       _ _
5  aytdi      ayt        VERB  _ _ 0  root      _ _

# sent_id = 12
# text = Talabalar qiyin imtihon topshirishdi
1  Talabalar     talaba   NOUN  _ _ 4  nsubj     _ _
2  qiyin         qiyin    ADJ   _ _ 3  amod      _ _
3  imtihon       imtihon  NOUN  _ _ 4  obj       _ _
4  topshirishdi  topshir  VERB  _ _ 0  root      _ _

# sent_id = 13
# text = Mushuk divanda uxlayapti
1  Mushuk     mushuk  NOUN  _ _ 3  nsubj     _ _
2  divanda    divan   NOUN  _ _ 3  obl       _ _
3  uxlayapti  uxla    VERB  _ _ 0  root      _ _

# sent_id = 14
# text = Bola masalani tez yechdi
1  Bola      bola    NOUN  _ _ 4  nsubj     _ _
2  masalani  masala  NOUN  _ _ 4  obj       _ _
3  tez       tez     ADV   _ _ 4  advmod    _ _
4  yechdi    yech    VERB  _ _ 0  root      _ _

# sent_id = 15
# text = Biz kitob oʻqiymiz
1  Biz       biz    PRON  _ _ 3  nsubj     _ _
2  kitob     kitob  NOUN  _ _ 3  obj       _ _
3  oʻqiymiz  oʻqi   VERB  _ _ 0  root      _ _

# sent_id = 16
# text = Shifokor kasal bolani koʻrdi
1  Shifokor  shifokor  NOUN  _ _ 4  nsubj     _ _
2  kasal     kasal     ADJ   _ _ 3  amod      _ _
3  bolani    bola      NOUN  _ _ 4  obj       _ _
4  koʻrdi    koʻr      VERB  _ _ 0  root      _ _

# sent_id = 17
# text = Ahmad doʻkondan non oldi
1  Ahmad      Ahmad   PROPN _ _ 4  nsubj     _ _
2  doʻkondan  doʻkon  NOUN  _ _ 4  obl       _ _
3  non        non     NOUN  _ _ 4  obj       _ _
4  oldi       ol      VERB  _ _ 0  root      _ _

# sent_id = 18
# text = Qiz onasiga chiroyli gullar berdi
1  Qiz       qiz       NOUN  _ _ 5  nsubj     _ _
2  onasiga   ona       NOUN  _ _ 5  obl       _ _
3  chiroyli  chiroyli  ADJ   _ _ 4  amod      _ _
4  gullar    gul       NOUN  _ _ 5  obj       _ _
5  berdi     ber       VERB  _ _ 0  root      _ _

# sent_id = 19
# text = Akamning kitobi stolda yotibdi
1  Akamning  aka    NOUN  _ _ 2  nmod:poss _ _
2  kitobi    kitob  NOUN  _ _ 4  nsubj     _ _
3  stolda    stol   NOUN  _ _ 4  obl       _ _
4  yotibdi   yot    VERB  _ _ 0  root      _ _
"""

def read_conllu(text):
    # блоки разделены пустой строкой; строки с # — комментарии; столбцы разделены пробелами
    sents = []
    for block in text.strip().split("\n\n"):
        toks = []
        for line in block.splitlines():
            if line.startswith("#") or not line.strip():
                continue
            c = line.split()
            toks.append(dict(id=int(c[0]), form=c[1], lemma=c[2], upos=c[3], head=int(c[6]), deprel=c[7]))
        sents.append(toks)
    return sents

train, test, other = read_conllu(TRAIN_CONLLU), read_conllu(TEST_CONLLU), read_conllu(OTHER_CONLLU)
print(f"обучение: {len(train)} предложений, тест: {len(test)} предложений\n")
print(pd.DataFrame(train[0])[["id", "form", "lemma", "upos", "head", "deprel"]].to_string(index=False))


def draw_tree(sent, ax, title=""):
    # дуга идёт от вершины к зависимому; красная — вершина справа, синяя — вершина слева
    n = len(sent)
    top = 0.6
    for t in sent:
        ax.text(t["id"], 0, t["form"], ha="center", va="top", fontsize=11)
        ax.text(t["id"], -0.45, t["upos"], ha="center", va="top", fontsize=8, color="gray")
    for t in sent:
        d, h = t["id"], t["head"]
        if h == 0:
            continue
        height = 0.3 + 0.45 * abs(h - d)
        top = max(top, height)
        theta = np.linspace(0, np.pi, 60)
        xs = (h + d) / 2 + (h - d) / 2 * np.cos(theta)
        ys = 0.15 + height * np.sin(theta)
        col = "#C0392B" if h > d else "#2B55C9"
        ax.plot(xs, ys, color=col, lw=1.5)
        ax.annotate("", xy=(d, 0.15), xytext=(xs[-4], ys[-4]), arrowprops=dict(arrowstyle="->", color=col, lw=1.5))
        ax.text((h + d) / 2, 0.15 + height, t["deprel"], ha="center", va="center", fontsize=9, color=col,
                bbox=dict(fc="white", ec="none", pad=0.5))
    root = [t["id"] for t in sent if t["head"] == 0][0]
    ax.annotate("", xy=(root, 0.15), xytext=(root, top + 0.6), arrowprops=dict(arrowstyle="->", color="black"))
    ax.text(root, top + 0.65, "root", ha="center", fontsize=9)
    ax.set_xlim(0.4, n + 0.6)
    ax.set_ylim(-0.9, top + 1.0)
    ax.axis("off")
    ax.set_title(title, fontsize=10)

fig, axes = plt.subplots(2, 1, figsize=(9, 6.5))
draw_tree(train[0], axes[0], "русский")
draw_tree(other[0], axes[1], "узбекский")
axes[0].plot([], [], color="#C0392B", label="вершина справа")
axes[0].plot([], [], color="#2B55C9", label="вершина слева")
fig.legend(loc="lower center", ncol=2, fontsize=9)
plt.tight_layout(rect=(0, 0.06, 1, 1))
plt.show()

В русском предложении глагол стоит в середине: подлежащее слева, дополнение и обстоятельство справа. В узбекском глагол стоит в конце (порядок SOV), и все дуги идут справа налево: язык последовательно **вершинно-финальный**. Посчитаем это по всему трибанку: для каждого типа связи — долю зависимых, стоящих перед вершиной.

Заодно проверим проективность: дерево проективно, если дуги можно нарисовать над предложением без пересечений. Метод arc-standard из раздела 2 строит только проективные деревья.

In [ ]:
def ancestors(sent, i):
    head = {t["id"]: t["head"] for t in sent}
    out = set()
    while i != 0:
        i = head[i]
        out.add(i)
    return out

def is_projective(sent):
    # дуга (h, d) проективна, если все слова между h и d зависят (прямо или косвенно) от h
    for t in sent:
        h, d = t["head"], t["id"]
        for k in range(min(h, d) + 1, max(h, d)):
            if h not in ancestors(sent, k):
                return False
    return True

bad = [" ".join(t["form"] for t in s) for s in train + test + other if not is_projective(s)]
print("все деревья проективны" if not bad else "непроективные: " + "; ".join(bad))

def direction_table(sents):
    rows = [(t["deprel"], t["id"] < t["head"]) for s in sents for t in s if t["head"] != 0]
    df = pd.DataFrame(rows, columns=["deprel", "before"])
    tab = df.groupby("deprel")["before"].agg(["mean", "size"])
    tab.loc["все"] = [df["before"].mean(), len(df)]
    return tab

tabs = {"русский": direction_table(train + test), "узбекский": direction_table(other)}
order = sorted((set(tabs["русский"].index) | set(tabs["узбекский"].index)) - {"все"}) + ["все"]
summary = pd.DataFrame({lang: [f"{tab.loc[r, 'mean']:.2f}  (n={int(tab.loc[r, 'size'])})" if r in tab.index else "—"
                               for r in order] for lang, tab in tabs.items()}, index=order)
print("\nдоля зависимых, стоящих ПЕРЕД своей вершиной:")
print(summary.to_string())

rels = ["nsubj", "obj", "obl", "amod", "case", "все"]
x = np.arange(len(rels))
plt.figure(figsize=(7, 3.2))
for k, (lang, tab) in enumerate(tabs.items()):
    vals = [tab["mean"].get(r, np.nan) for r in rels]
    plt.bar(x + (k - 0.5) * 0.38, vals, 0.38, label=lang, color=["#2B55C9", "#E08E2B"][k])
plt.xticks(x, rels)
plt.ylim(0, 1.05)
plt.ylabel("доля")
plt.title("Где стоит зависимое: доля случаев «перед вершиной»", fontsize=10)
plt.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

Цифры подтверждают типологию. В узбекском почти всё стоит перед вершиной; исключение — `case`: послелог `bilan` стоит после существительного, а в UD служебное слово считается зависимым. В русском `obj` и `obl` стоят после глагола, а предлоги (`case`) — перед существительным. Прилагательные (`amod`) в обоих языках стоят перед существительным.

`nltk` умеет читать CoNLL-U в объект `DependencyGraph` и печатать дерево:

In [ ]:
from nltk.parse import DependencyGraph

def to_conllu_text(sent, heads=None):
    # 10 столбцов CoNLL-U через табуляцию; heads — словарь {id: (вершина, тип)}, если дерево предсказано парсером
    lines = []
    for t in sent:
        h, r = heads[t["id"]] if heads else (t["head"], t["deprel"])
        lines.append("\t".join([str(t["id"]), t["form"], t["lemma"], t["upos"], "_", "_", str(h), r, "_", "_"]))
    return "\n".join(lines)

g = DependencyGraph(to_conllu_text(train[0]), top_relation_label="root")
g.tree().pretty_print()
for (h, h_pos), rel, (d, d_pos) in g.triples():
    print(f"   {h:<14} --{rel}--> {d}")

## 2. Парсер arc-standard

Переходный (transition-based) парсер читает предложение слева направо и хранит **стек** обработанных слов и **буфер** ещё не прочитанных. На каждом шаге он делает одно из действий:

- **SHIFT** — перенести первое слово буфера на стек;
- **LEFT-ARC:r** — второе сверху слово стека становится зависимым верхнего (дуга влево), снимаем его со стека;
- **RIGHT-ARC:r** — верхнее слово становится зависимым второго (дуга вправо), снимаем верхнее.

**Оракул** по эталонному дереву говорит, какое действие правильное: LEFT-ARC, если второе слово зависит от верхнего; RIGHT-ARC, если верхнее зависит от второго и уже собрало всех своих зависимых; иначе SHIFT. Для предложения из $n$ слов действий ровно $2n$: каждое слово один раз попадает на стек и один раз с него снимается.

In [ ]:
def apply(action, stack, buffer, arcs):
    stack, buffer, arcs = list(stack), list(buffer), list(arcs)
    if action == "SHIFT":
        stack.append(buffer.pop(0))
    elif action.startswith("LEFT-ARC"):
        dep = stack.pop(-2)
        arcs.append((stack[-1], dep, action.split(":", 1)[1]))
    else:
        dep = stack.pop()
        arcs.append((stack[-1], dep, action.split(":", 1)[1]))
    return stack, buffer, arcs

def oracle(sent):
    # статический оракул: по эталонному дереву говорит, какое действие сделать
    head = {t["id"]: t["head"] for t in sent}
    rel = {t["id"]: t["deprel"] for t in sent}
    n_deps = Counter(head.values())
    stack, buffer, arcs = [0], [t["id"] for t in sent], []
    steps = []
    while buffer or len(stack) > 1:
        done = Counter(h for h, _, _ in arcs)
        if len(stack) > 2 and head[stack[-2]] == stack[-1]:
            action = "LEFT-ARC:" + rel[stack[-2]]
        elif len(stack) >= 2 and head[stack[-1]] == stack[-2] and done[stack[-1]] == n_deps[stack[-1]]:
            action = "RIGHT-ARC:" + rel[stack[-1]]
        elif buffer:
            action = "SHIFT"
        else:
            raise ValueError("непроективное дерево: оракул не может построить его")
        steps.append((stack, buffer, arcs, action))
        stack, buffer, arcs = apply(action, stack, buffer, arcs)
    return steps, arcs

def show_trace(sent):
    form = {0: "ROOT", **{t["id"]: t["form"] for t in sent}}
    steps, arcs = oracle(sent)
    rows = []
    for k, (st, bf, ar, action) in enumerate(steps, 1):
        new = apply(action, st, bf, ar)[2][len(ar):]
        arc = f"{form[new[0][0]]} → {form[new[0][1]]}" if new else ""
        rows.append([k, " ".join(form[i] for i in st), " ".join(form[i] for i in bf), action, arc])
    print(pd.DataFrame(rows, columns=["шаг", "стек", "буфер", "действие", "новая дуга"]).to_string(index=False))
    gold = sorted((t["head"], t["id"], t["deprel"]) for t in sent)
    print("\nдерево, построенное оракулом, совпадает с эталоном:", sorted(arcs) == gold)

show_trace(train[0])

Сравним русский и узбекский. В вершинно-финальном языке вершина приходит последней, поэтому парсер долго делает SHIFT и складывает зависимые на стек, а потом снимает их серией LEFT-ARC. Стек получается глубже.

In [ ]:
def short(action):
    return {"SHIFT": "S", "LEFT-ARC": "L", "RIGHT-ARC": "R"}[action.split(":")[0]]

for lang, sents in [("русский", train), ("узбекский", other)]:
    depths = [max(len(st) for st, _, _, _ in oracle(s)[0]) - 1 for s in sents]
    print(f"{lang:<10} максимальная глубина стека (без ROOT), среднее по предложениям: {np.mean(depths):.2f}")
    for s in sents[:3]:
        steps, _ = oracle(s)
        print(f"   {' '.join(t['form'] for t in s):<46} {' '.join(short(a) for *_, a in steps)}")
    print()

Оракулу нужен эталонный ответ. Настоящий парсер должен угадывать действие по текущей конфигурации, не зная дерева. Превратим трассы оракула в обучающие примеры «признаки конфигурации → действие» и обучим логистическую регрессию. Такую схему (жадный парсер с классификатором) предложил Й. Нивре; в современных парсерах вместо логистической регрессии стоит нейросеть, но цикл тот же.

In [ ]:
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

def config_features(stack, buffer, arcs, sent):
    # признаки конфигурации: части речи и окончания двух верхних слов стека и первого слова буфера, уже присоединённые левые зависимые
    tok = {t["id"]: t for t in sent}
    def upos(i):
        return "<none>" if i is None else ("ROOT" if i == 0 else tok[i]["upos"])
    def suf(i):
        return "<none>" if i is None else ("ROOT" if i == 0 else tok[i]["form"].lower()[-2:])
    s0 = stack[-1] if stack else None
    s1 = stack[-2] if len(stack) >= 2 else None
    b0 = buffer[0] if buffer else None
    left = sorted(r for h, d, r in arcs if h == s0 and d < s0)
    return {
        "s0.upos": upos(s0), "s1.upos": upos(s1), "b0.upos": upos(b0),
        "s1+s0": upos(s1) + "+" + upos(s0),
        "s0.suf": suf(s0), "s1.suf": suf(s1),
        "s0.left": "|".join(left) or "-",
    }

X, y = [], []
for s in train:
    for stack, buffer, arcs, action in oracle(s)[0]:
        X.append(config_features(stack, buffer, arcs, s))
        y.append(action)
vec = DictVectorizer()
clf = LogisticRegression(max_iter=3000, C=10).fit(vec.fit_transform(X), y)
print(f"{len(X)} обучающих примеров (конфигурация → действие)")
print(pd.Series(y).value_counts().to_string())

Парсер работает жадно: на каждом шаге берёт самое вероятное **допустимое** действие и не возвращается назад, как жадный декодер в практикуме 8. Качество измеряют долей слов с правильной вершиной (UAS, unlabeled attachment score) и с правильной вершиной и типом связи (LAS, labeled). Части речи считаем известными (их дал бы теггер из практикума 8). Знаки: ✓ — всё верно, ~ — верна вершина, но не тип, ✗ — неверна вершина.

In [ ]:
def legal(action, stack, buffer):
    # действие допустимо? LEFT-ARC нельзя применить к ROOT; RIGHT-ARC к ROOT — только когда буфер пуст
    if action == "SHIFT":
        return bool(buffer)
    if action.startswith("LEFT-ARC"):
        return len(stack) > 2
    return len(stack) >= 2 and (stack[-2] != 0 or not buffer)

def parse(sent):
    stack, buffer, arcs = [0], [t["id"] for t in sent], []
    while buffer or len(stack) > 1:
        probs = clf.predict_proba(vec.transform([config_features(stack, buffer, arcs, sent)]))[0]
        for k in np.argsort(-probs):
            if legal(clf.classes_[k], stack, buffer):
                break
        stack, buffer, arcs = apply(clf.classes_[k], stack, buffer, arcs)
    return {d: (h, r) for h, d, r in arcs}

uas = las = n = 0
for s in test:
    pred = parse(s)
    form = {0: "ROOT", **{t["id"]: t["form"] for t in s}}
    print(" ".join(t["form"] for t in s))
    for t in s:
        h, r = pred[t["id"]]
        mark = "✓" if (h, r) == (t["head"], t["deprel"]) else ("~" if h == t["head"] else "✗")
        print(f"   {mark} {t['form']:<14} эталон: {form[t['head']]:<12} {t['deprel']:<10} парсер: {form[h]:<12} {r}")
        uas += h == t["head"]
        las += (h, r) == (t["head"], t["deprel"])
        n += 1
print(f"\nUAS = {uas}/{n} = {uas / n:.2f}   LAS = {las}/{n} = {las / n:.2f}")

last = test[-1]
DependencyGraph(to_conllu_text(last, parse(last)), top_relation_label="root").tree().pretty_print()

Посмотрите на ошибку типа связи. В обучении косвенные дополнения (`iobj`, дательный падеж) кончались на «-ам», «-ям», «-е» («ученикам», «детям», «подруге»), а на «-у» кончались прямые дополнения в винительном падеже («задачу», «книгу» → `obj`). Поэтому «директору» парсер счёл прямым дополнением: падежа он не знает, только последние буквы. Морфологические признаки (падеж из `pymorphy3`, практикум 3) решили бы эту проблему.

## 3. Деревья составляющих и алгоритм CYK

Дерево составляющих (constituency) группирует слова в вложенные фразы: именная группа NP, глагольная VP, предложная PP. Правила задаёт контекстно-свободная грамматика (CFG). Алгоритм CYK требует **нормальную форму Хомского** (CNF): каждое правило имеет вид $A \to B\,C$ или $A \to \text{слово}$.

CYK заполняет таблицу снизу вверх: ячейка $[i, j]$ хранит все нетерминалы, из которых выводятся слова $i..j-1$. Для отрезков длины 1 смотрим в словарь, для длинных перебираем точку разреза $k$: если $B \in [i,k]$, $C \in [k,j]$ и есть правило $A \to B\,C$, добавляем $A$ в $[i,j]$ с обратной ссылкой. Сложность $O(n^3 \cdot |G|)$. Это то же динамическое программирование, что в Витерби, только по отрезкам, а не по позициям.

Пример из лаборатории: «я видел девушку с биноклем». Кто держит бинокль?

После печати шагов таблица рисуется картинкой: в верхней правой ячейке — S, причём двумя способами (×2).

In [ ]:
import nltk

GRAMMAR_TEXT = """
S  -> NP VP
VP -> V NP | VP PP
NP -> NP PP | 'я' | 'девушку' | 'биноклем' | 'парке' | 'озера' | 'городом'
PP -> P NP
V  -> 'видел'
P  -> 'с' | 'в' | 'у' | 'за'
"""

def read_grammar(text):
    # разбираем правила вида A -> B C | 'слово'
    binary, lexical = defaultdict(list), defaultdict(list)
    for line in text.strip().splitlines():
        lhs, rhs = [x.strip() for x in line.split("->")]
        for alt in rhs.split("|"):
            parts = alt.split()
            if len(parts) == 1 and parts[0][0] in "'\"":
                lexical[parts[0].strip("'\"")].append(lhs)
            else:
                binary[tuple(parts)].append(lhs)
    return binary, lexical

binary, lexical = read_grammar(GRAMMAR_TEXT)
print(f"грамматика в нормальной форме Хомского: {nltk.CFG.fromstring(GRAMMAR_TEXT).is_chomsky_normal_form()}; "
      f"{sum(map(len, binary.values()))} бинарных правил, {len(lexical)} слов в словаре\n")

def cyk(words, verbose=False):
    n = len(words)
    chart = defaultdict(lambda: defaultdict(list))     # chart[i, j][A] = [(k, B, C), ...]
    for i, w in enumerate(words):
        for A in lexical.get(w, []):
            chart[i, i + 1][A].append(w)
        if verbose:
            print(f"ячейка [{i},{i + 1}] {w}: {', '.join(chart[i, i + 1])}")
    for length in range(2, n + 1):
        for i in range(n - length + 1):
            j = i + length
            for k in range(i + 1, j):
                for B in list(chart[i, k]):
                    for C in list(chart[k, j]):
                        for A in binary.get((B, C), []):
                            chart[i, j][A].append((k, B, C))
            if verbose and chart[i, j]:
                found = "; ".join(f"{A} ← " + " | ".join(f"{B}[{i},{k}] + {C}[{k},{j}]" for k, B, C in bps)
                                  for A, bps in chart[i, j].items())
                print(f"ячейка [{i},{j}] {' '.join(words[i:j])}: {found}")
    return chart

def count(chart, i, j, A, memo=None):
    memo = {} if memo is None else memo
    if (i, j, A) not in memo:
        total = 0
        for bp in chart[i, j][A]:
            if isinstance(bp, str):
                total += 1
            else:
                k, B, C = bp
                total += count(chart, i, k, B, memo) * count(chart, k, j, C, memo)
        memo[i, j, A] = total
    return memo[i, j, A]

words = "я видел девушку с биноклем".split()
chart = cyk(words, verbose=True)
print(f"\nS [0,{len(words)}]: {count(chart, 0, len(words), 'S')} разборов")


n = len(words)
fig, ax = plt.subplots(figsize=(1.6 * n + 1, 0.9 * n + 1))
for i in range(n):
    for j in range(i + 1, n + 1):
        cell = chart[i, j]
        txt = "\n".join(f"{A}" + (f" ×{count(chart, i, j, A)}" if count(chart, i, j, A) > 1 else "") for A in cell)
        ax.add_patch(plt.Rectangle((j - 1.5, i - 0.5), 1, 1, fill=bool(cell), fc="#DCE4F7", ec="gray"))
        ax.text(j - 1, i, txt, ha="center", va="center", fontsize=9)
ax.set_xlim(-0.5, n - 0.5)
ax.set_ylim(n - 0.5, -0.5)
ax.set_xticks(range(n), words)
ax.set_yticks(range(n), [f"{i}: {w}" for i, w in enumerate(words)])
ax.set_xlabel("последнее слово")
ax.set_ylabel("начало")
ax.set_title("Таблица CYK: ячейка [i, j] — составляющие, покрывающие слова i..j-1 (×k — k способов)", fontsize=9)
plt.tight_layout()
plt.show()

По обратным ссылкам восстановим все деревья и сравним с `nltk.ChartParser` — библиотечным парсером для произвольных CFG. Деревья печатает `nltk.Tree.pretty_print()`.

Первое дерево: PP «с биноклем» присоединена к VP — смотрел с помощью бинокля. Второе: PP внутри NP — девушка с биноклем.

In [ ]:
def build_trees(chart, i, j, A):
    out = []
    for bp in chart[i, j][A]:
        if isinstance(bp, str):
            out.append(nltk.Tree(A, [bp]))
        else:
            k, B, C = bp
            for left in build_trees(chart, i, k, B):
                for right in build_trees(chart, k, j, C):
                    out.append(nltk.Tree(A, [left, right]))
    return out

ours = build_trees(chart, 0, len(words), "S")
grammar = nltk.CFG.fromstring(GRAMMAR_TEXT)
theirs = list(nltk.ChartParser(grammar).parse(words))
print(f"наш CYK: {len(ours)} разборов;  nltk.ChartParser: {len(theirs)} разборов;  "
      f"множества деревьев совпадают: {set(map(str, ours)) == set(map(str, theirs))}")
for t in theirs:
    t.pretty_print()

## 4. Рост неоднозначности и вероятностная грамматика

Каждая новая предложная группа может присоединиться к любой из предыдущих NP или к VP. Число разборов растёт как числа Каталана: 1, 2, 5, 14, 42… Для реальной грамматики и предложения из 30 слов разборов могут быть тысячи, поэтому нужен способ выбрать один.

**PCFG** (вероятностная КС-грамматика) приписывает каждому правилу вероятность, вероятность дерева — произведение вероятностей его правил. `nltk.ViterbiParser` находит самое вероятное дерево тем же CYK, где вместо множества нетерминалов в ячейке хранится максимум вероятности.

In [ ]:
for s in [
    "я видел девушку",
    "я видел девушку с биноклем",
    "я видел девушку с биноклем в парке",
    "я видел девушку с биноклем в парке у озера",
    "я видел девушку с биноклем в парке у озера за городом",
]:
    w = s.split()
    ch = cyk(w)
    n_ours = count(ch, 0, len(w), "S")
    n_nltk = len(list(nltk.ChartParser(grammar).parse(w)))
    print(f"{len(w):2d} слов  {n_ours:4d} разборов (nltk: {n_nltk:3d})   {s}")

pcfg = nltk.PCFG.fromstring("""
S  -> NP VP        [1.0]
VP -> V NP         [0.7]
VP -> VP PP        [0.3]
NP -> NP PP        [0.2]
NP -> 'я'          [0.2]
NP -> 'девушку'    [0.2]
NP -> 'биноклем'   [0.1]
NP -> 'парке'      [0.1]
NP -> 'озера'      [0.1]
NP -> 'городом'    [0.1]
PP -> P NP         [1.0]
V  -> 'видел'      [1.0]
P  -> 'с' [0.4] | 'в' [0.3] | 'у' [0.2] | 'за' [0.1]
""")
prob = {(p.lhs(), p.rhs()): p.prob() for p in pcfg.productions()}
print()
for t in theirs:
    p = np.prod([prob[(r.lhs(), r.rhs())] for r in t.productions()])
    print(f"вероятность разбора = {p:.6f}:  {t.pformat(margin=200)}")
best = next(nltk.ViterbiParser(pcfg).parse(words))
print(f"\nViterbiParser выбирает (p = {best.prob():.6f}):")
best.pretty_print()

Два дерева отличаются одним правилом: VP → VP PP (0,3) против NP → NP PP (0,2). Поэтому PCFG всегда выберет присоединение к глаголу, какие бы слова ни стояли в предложении: и в «видел девушку с биноклем», и в «видел девушку с косичками». PCFG не смотрит на слова при выборе структуры. Лексикализованные грамматики и нейросетевые парсеры учитывают, какие слова соединяются.

## 5. Задания

1. **Свой трибанк.** Запишите в формате CoNLL-U два новых предложения (одно — с `nmod`, например «Дом моего друга стоит у реки»). Проверьте `is_projective`, нарисуйте `draw_tree` и запустите `show_trace`.
2. **Непроективность.** Разметьте предложение «Интересную я вчера прочитал книгу» (`amod` от «Интересную» к «книгу»). Убедитесь, что `is_projective` возвращает `False` и что `oracle` останавливается с ошибкой. Какие дуги пересекаются?
3. **Признаки парсера.** Добавьте в `config_features` лемму верхнего слова стека и окончание первого слова буфера. Переобучите классификатор и сравните UAS и LAS. Исчезла ли путаница `obj` / `iobj`?
4. **Своя грамматика.** Добавьте в `GRAMMAR_TEXT` прилагательные (правила `NP -> Adj NP`, `Adj -> 'красивую'`) и разберите «я видел красивую девушку с биноклем». Сколько разборов? Остаётся ли грамматика в CNF?
5. **PCFG.** Подберите вероятности `VP -> VP PP` и `NP -> NP PP` (сумма вероятностей правил с одной левой частью должна оставаться 1), чтобы `ViterbiParser` выбирал присоединение к NP. При каком соотношении вероятностей разборы становятся равновероятными?

*Совет: можно взять настоящий трибанк Universal Dependencies (например, файл `ru_syntagrus-ud-dev.conllu`), загрузить его через панель «Файлы» и прочитать функцией `read_conllu(open(...).read())`.*

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь

In [ ]:
# Ваш код здесь